<a href="https://colab.research.google.com/github/agharikrishnan/Flyrank_ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/agharikrishnan/Flyrank_ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*I want to prioritize pages that already have some search visibility and may be worth reviewing for a refresh. The visibility signal looks useful because pages with more impressions generally have more sessions. Staleness is more mixed: the 91–180 day group looks useful, but the 181+ group has very low traffic, so I don't want to treat every old page as a priority. My rule will therefore give more weight to visibility and use moderate staleness as a supporting signal. The reason code will be visible_stale, and the action will be review_refresh.*

In [5]:
from pathlib import Path
import pandas as pd

# Find the starter CSV anywhere under /content
matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))

csv_path = matches[0]
df = pd.read_csv(csv_path)
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")

Rows: 30,000
Columns: 44


## 2. Build the ranked queue (writes the CSV)

*I’m keeping the rule simple so it is easy to understand why a page gets a high score. I give more points to pages with higher impressions because the signal check showed that pages with more visibility also tend to have more sessions. I give one extra point to pages that have not been updated for 91 to 180 days. I’m not giving extra points to pages older than 180 days because that group had very low impressions and sessions. Pages with a positive score will be marked for review_refresh.*

In [10]:
import numpy as np
from pathlib import Path

df["visibility_points"] = pd.cut(
    df["impressions_90d"],
    bins=[-1, 100, 500, 1000, 5000, float("inf")],
    labels=[0, 1, 2, 3, 4]
).astype(int)

df["staleness_points"] = (
    df["days_since_last_update"].between(91, 180)
).astype(int)

df["score"] = (
    df["visibility_points"] +
    df["staleness_points"]
)

df["reason_code"] = np.where(
    df["score"] > 0,
    "visible_stale",
    "low_priority"
)

df["action"] = np.where(
    df["score"] > 0,
    "review_refresh",
    "monitor"
)

df = df.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

output = df[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "impressions_90d",
        "days_since_last_update"
    ]
].copy()

output_path = Path(
    "/content/Flyrank_ML/work/outputs/baseline_action_score.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(output_path, index=False)

print("Rows ranked:", len(output))
print("CSV written to:", output_path)

display(output.head(10))

Rows ranked: 30000
CSV written to: /content/Flyrank_ML/work/outputs/baseline_action_score.csv


,rank,content_id,score,reason_code,action,impressions_90d,days_since_last_update
0,1,content_5fe46e04994d,5,visible_stale,review_refresh,517715,104
1,2,content_2dba2b1f9536,5,visible_stale,review_refresh,443434,104
2,3,content_2c2606c5d176,5,visible_stale,review_refresh,347399,104
3,4,content_cb112fce36be,5,visible_stale,review_refresh,309910,104
4,5,content_9532f197bbc8,5,visible_stale,review_refresh,309192,104
5,6,content_36ff89c8214e,5,visible_stale,review_refresh,295097,104
6,7,content_b28d1efd668f,5,visible_stale,review_refresh,286608,104
7,8,content_813e88069237,5,visible_stale,review_refresh,233561,104
8,9,content_c21024970297,5,visible_stale,review_refresh,211366,104
9,10,content_c8e9d6ab9013,5,visible_stale,review_refresh,208678,104


## 3. Top-20 review

*I looked at the top 20 pages to see if the rule is actually giving reasonable results. For each page, I recorded the action, the reason it was ranked highly, and what could make the recommendation wrong. This is important because a high score does not automatically mean that a page needs to be refreshed. A human review is still needed before taking action*

In [11]:
top20 = output.head(20).copy()

top20["confidence_note"] = np.where(
    top20["score"] >= 4,
    "strong signal",
    "moderate signal"
)

top20["what_would_make_it_wrong"] = np.where(
    top20["days_since_last_update"] > 180,
    "very old page with weak current demand",
    np.where(
        top20["impressions_90d"] < 500,
        "low search visibility",
        "human review may show that a refresh is not needed"
    )
)

review = top20[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

display(review)

,rank,content_id,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,content_5fe46e04994d,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...
1,2,content_2dba2b1f9536,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...
2,3,content_2c2606c5d176,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...
3,4,content_cb112fce36be,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...
4,5,content_9532f197bbc8,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...
5,6,content_36ff89c8214e,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...
6,7,content_b28d1efd668f,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...
7,8,content_813e88069237,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...
8,9,content_c21024970297,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...
9,10,content_c8e9d6ab9013,5,visible_stale,review_refresh,strong signal,human review may show that a refresh is not ne...


## 4. Weak picks + leakage check

*A few of the top picks can still be wrong. High impressions tell me that a page is visible, but they do not tell me that a refresh will definitely improve it. A page may already be performing well, or there may be another reason why it should not be changed. I also checked the rule inputs to make sure I did not use the decline label or any future-window information. The score only uses impressions_90d and days_since_last_update.*

In [12]:

weak_picks = output[
    (output["score"] <= 2)
].head(10).copy()

print("Example weaker picks:")
display(weak_picks)

print("\nLeakage check:")

score_features = [
    "impressions_90d",
    "days_since_last_update"
]

print("Score features:", score_features)

leakage_fields = [
    "trend_direction",
    "trend_pct"
]

used_leakage_fields = [
    col for col in leakage_fields
    if col in score_features
]

print("Label-derived fields used:", used_leakage_fields)

if len(used_leakage_fields) == 0:
    print("PASS: no label-derived fields were used in the score.")
else:
    print("CHECK NEEDED: label-derived field found.")

Example weaker picks:


,rank,content_id,score,reason_code,action,impressions_90d,days_since_last_update
14597,14598,content_5dd03d9f4866,2,visible_stale,review_refresh,1000,11
14598,14599,content_9b154ec7af5a,2,visible_stale,review_refresh,1000,22
14599,14600,content_84277809a7de,2,visible_stale,review_refresh,1000,22
14600,14601,content_ff71027b45c6,2,visible_stale,review_refresh,999,22
14601,14602,content_cde9fc17927f,2,visible_stale,review_refresh,999,7
14602,14603,content_9aeec226c5af,2,visible_stale,review_refresh,998,11
14603,14604,content_4f3e4a193869,2,visible_stale,review_refresh,998,20
14604,14605,content_e36c8972e97a,2,visible_stale,review_refresh,998,22
14605,14606,content_b4bb27d63f25,2,visible_stale,review_refresh,998,20
14606,14607,content_935fb7b7aa07,2,visible_stale,review_refresh,998,22



Leakage check:
Score features: ['impressions_90d', 'days_since_last_update']
Label-derived fields used: []
PASS: no label-derived fields were used in the score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.